# Week 2 Step 6 — Model Selection: Haiku 3.5 vs GPT-4o mini

**Goal:** Choose the synthesis LLM for CPIE v1 based on quality, JSON compliance, latency, and cost.

**Corpus:** 3 pilot PDFs (Ofgem SSES 2025, CBES Results 2022, IEA WEO 2025)  
**Retrieval:** Hybrid RRF (BM25 + bge-base-en-v1.5) → cross-encoder reranker → top-5 chunks  
**Queries:** 10 total — 6 factual, 2 cross-document synthesis, 2 out-of-corpus negatives  

**Decision criteria (in order):**
1. JSON schema compliance (non-negotiable — schema enforced at runtime)
2. Answer quality: accuracy + citation faithfulness (manual 1–5 score)
3. Out-of-corpus handling: should return confidence < 0.4 and honest answer for negatives
4. Cost per query (secondary — both models are cheap)
5. Latency (tertiary — both acceptable for async use)

In [1]:
import json
import logging
import os
import re
import statistics
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = str(ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

# Pre-load heavy native libs explicitly before retrieval triggers them
# indirectly — avoids kernel crash from deep import chain on Windows
import torch
import pyarrow
import sentence_transformers
import chromadb

import fitz
import tiktoken
from dotenv import load_dotenv

from retrieval import BM25Retriever, DenseRetriever, HybridRetriever, Reranker

logging.basicConfig(level=logging.WARNING)
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Project root: {ROOT}")
print("Imports OK.")

torch 2.6.0+cu124 | CUDA: True
Project root: C:\AI_Engineering\cpie
Imports OK.


## 1. API clients

In [2]:
load_dotenv(ROOT / '.env')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')
OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY', '')

assert ANTHROPIC_API_KEY, 'ANTHROPIC_API_KEY not set — add it to .env'
assert OPENAI_API_KEY,    'OPENAI_API_KEY not set — add it to .env'

import anthropic
import openai

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client    = openai.OpenAI(api_key=OPENAI_API_KEY)

print('API clients ready.')

API clients ready.


## 2. Build pilot corpus

Same chunking config as `scripts/validate_retrieval.py` (locked Week 2 settings).

In [3]:
CHUNK_SIZE = 400
OVERLAP    = 80
MIN_TOKENS = 50
TOKENIZER  = tiktoken.get_encoding('cl100k_base')

PILOT_DOCS = [
    {
        'path': ROOT / 'data/raw/Smart-Secure-Electricity-Systems-Implementing-the-load-control-licensing-regime-consultation.pdf',
        'doc_id': 'OFGEM_SSES_2025',
        'institution': 'Ofgem',
        'doc_type': 'consultation',
        'jurisdiction': 'UK',
        'publication_date': '2025',
    },
    {
        'path': ROOT / 'data/raw/results-of-the-2021-climate-biennial-exploratory-scenario.pdf',
        'doc_id': 'BOE_CBES_RESULTS_2022',
        'institution': 'FCA/PRA',
        'doc_type': 'results_report',
        'jurisdiction': 'UK',
        'publication_date': '2022',
    },
    {
        'path': ROOT / 'data/raw/WorldEnergyOutlook2025.pdf',
        'doc_id': 'IEA_WEO_2025',
        'institution': 'IEA',
        'doc_type': 'outlook_report',
        'jurisdiction': 'Global',
        'publication_date': '2025',
    },
]


def extract_text(pdf_path: Path) -> str:
    doc = fitz.open(str(pdf_path))
    text = '\n'.join(page.get_text('text') for page in doc)
    doc.close()
    return text


def chunk_text(text: str, doc_meta: dict) -> list[dict]:
    tokens = TOKENIZER.encode(text)
    step   = CHUNK_SIZE - OVERLAP
    chunks = []
    for start in range(0, len(tokens), step):
        token_slice = tokens[start: start + CHUNK_SIZE]
        if len(token_slice) < MIN_TOKENS:
            break
        chunks.append({
            'text':             TOKENIZER.decode(token_slice),
            'doc_id':           doc_meta['doc_id'],
            'institution':      doc_meta['institution'],
            'doc_type':         doc_meta['doc_type'],
            'jurisdiction':     doc_meta['jurisdiction'],
            'publication_date': doc_meta['publication_date'],
            'page_number':      -1,
            'chunk_index':      len(chunks),
            'token_count':      len(token_slice),
            'chunk_type':       'prose',
        })
    return chunks


all_chunks: list[dict] = []
for meta in PILOT_DOCS:
    if not meta['path'].exists():
        print(f'  WARNING: PDF not found: {meta["path"].name}')
        continue
    text   = extract_text(meta['path'])
    chunks = chunk_text(text, meta)
    print(f'  {meta["doc_id"]}: {len(chunks)} chunks')
    all_chunks.extend(chunks)

print(f'\nTotal corpus: {len(all_chunks)} chunks across {len(PILOT_DOCS)} docs')

  OFGEM_SSES_2025: 98 chunks
  BOE_CBES_RESULTS_2022: 100 chunks
  IEA_WEO_2025: 998 chunks

Total corpus: 1196 chunks across 3 docs


## 3. Build retrieval pipeline

Dense encoding takes ~1–2 min on the RTX 4050 for this corpus size.

In [4]:
# BM25
print('Building BM25 index...')
bm25 = BM25Retriever(k1=1.5, b=0.75)
bm25.build(all_chunks)
print(f'  BM25 ready ({len(all_chunks)} chunks)')

# Dense (will encode on GPU)
print('\nBuilding dense index (encoding on GPU — ~1-2 min)...')
dense = DenseRetriever(
    persist_dir=ROOT / 'data/processed/chroma_db_model_select',
    collection_name='cpie_model_select',
)
dense.build(all_chunks)
print('  Dense index ready')

# Hybrid + reranker
hybrid   = HybridRetriever(bm25=bm25, dense=dense, rrf_k=60)
reranker = Reranker(top_k=5)
print('\nRetrieval pipeline ready (BM25 + dense + RRF k=60 + cross-encoder reranker)')

Building BM25 index...
  BM25 ready (1196 chunks)

Building dense index (encoding on GPU — ~1-2 min)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

  Dense index ready

Retrieval pipeline ready (BM25 + dense + RRF k=60 + cross-encoder reranker)


## 4. Evaluation queries

10 queries: 6 factual (2 per doc), 2 cross-document synthesis, 2 out-of-corpus negatives.

In [5]:
EVAL_QUERIES = [
    # Factual — Ofgem SSES 2025
    {
        'id': 'Q01', 'type': 'factual', 'expected_source': 'OFGEM_SSES_2025',
        'query': 'What load control licensing requirements does Ofgem propose for flexibility service providers?',
    },
    {
        'id': 'Q02', 'type': 'factual', 'expected_source': 'OFGEM_SSES_2025',
        'query': 'What enforcement powers will Ofgem have against unlicensed load control operators?',
    },
    # Factual — CBES Results 2022
    {
        'id': 'Q03', 'type': 'factual', 'expected_source': 'BOE_CBES_RESULTS_2022',
        'query': 'What aggregate loan losses did UK banks face under the CBES early action scenario?',
    },
    {
        'id': 'Q04', 'type': 'factual', 'expected_source': 'BOE_CBES_RESULTS_2022',
        'query': 'How did the CBES late policy transition scenario affect UK bank profitability and capital ratios?',
    },
    # Factual — IEA WEO 2025
    {
        'id': 'Q05', 'type': 'factual', 'expected_source': 'IEA_WEO_2025',
        'query': 'What does the IEA project for peak global fossil fuel demand in its Stated Policies Scenario?',
    },
    {
        'id': 'Q06', 'type': 'factual', 'expected_source': 'IEA_WEO_2025',
        'query': 'What annual clean energy investment does the IEA project will be needed globally by 2030?',
    },
    # Cross-document synthesis
    {
        'id': 'Q07', 'type': 'synthesis', 'expected_source': None,
        'query': 'How do Ofgem load control licensing proposals relate to the energy transition goals described in the IEA World Energy Outlook?',
    },
    {
        'id': 'Q08', 'type': 'synthesis', 'expected_source': None,
        'query': 'What do the CBES bank stress test results imply about UK financial sector readiness for the clean energy investment the IEA projects?',
    },
    # Out-of-corpus negatives
    {
        'id': 'Q09', 'type': 'negative', 'expected_source': None,
        'query': 'What are the EU taxonomy green bond standards for small modular reactor financing?',
    },
    {
        'id': 'Q10', 'type': 'negative', 'expected_source': None,
        'query': 'What Basel III Pillar 2 capital add-ons apply to fossil fuel lending exposures under EBA guidelines?',
    },
]

print(f'{len(EVAL_QUERIES)} evaluation queries:\n')
for q in EVAL_QUERIES:
    src = q['expected_source'] or 'N/A (cross-doc / out-of-corpus)'
    print(f"  {q['id']} [{q['type']:9s}] {q['query'][:65]}")
    print(f"             expected: {src}")

10 evaluation queries:

  Q01 [factual  ] What load control licensing requirements does Ofgem propose for f
             expected: OFGEM_SSES_2025
  Q02 [factual  ] What enforcement powers will Ofgem have against unlicensed load c
             expected: OFGEM_SSES_2025
  Q03 [factual  ] What aggregate loan losses did UK banks face under the CBES early
             expected: BOE_CBES_RESULTS_2022
  Q04 [factual  ] How did the CBES late policy transition scenario affect UK bank p
             expected: BOE_CBES_RESULTS_2022
  Q05 [factual  ] What does the IEA project for peak global fossil fuel demand in i
             expected: IEA_WEO_2025
  Q06 [factual  ] What annual clean energy investment does the IEA project will be 
             expected: IEA_WEO_2025
  Q07 [synthesis] How do Ofgem load control licensing proposals relate to the energ
             expected: N/A (cross-doc / out-of-corpus)
  Q08 [synthesis] What do the CBES bank stress test results imply about UK financia
         

## 5. Synthesis prompt & helpers

In [6]:
SYNTHESIS_PROMPT = """You are a climate policy analyst assistant. Using ONLY the provided document excerpts, answer the query below.

Your response must be valid JSON with exactly this structure:
{{
  "answer": "<direct answer to the query, 2-4 sentences>",
  "citations": [
    {{"doc_id": "<document identifier>", "passage": "<verbatim excerpt, max 60 words>", "page": -1}}
  ],
  "confidence": <float 0.0-1.0>,
  "contradictions": []
}}

Rules:
- If the excerpts do not contain sufficient information to answer, set confidence below 0.3 and set answer to: "The available documents do not contain sufficient information to answer this query."
- Do not invent or infer facts not present in the excerpts.
- Include 1-3 citations from the most relevant excerpts only.
- Set confidence >= 0.7 only when excerpts directly and clearly support the answer.
- Respond with valid JSON only. No preamble, no markdown fences.

Query: {query}

Document excerpts:
{context}"""


def format_context(chunks: list[dict], max_chars: int = 600) -> str:
    """Format reranked chunks into a context block for the prompt."""
    parts = []
    for i, chunk in enumerate(chunks, 1):
        parts.append(
            f"[Excerpt {i} | doc_id: {chunk['doc_id']} | "
            f"institution: {chunk['institution']} | date: {chunk['publication_date']}]\n"
            f"{chunk['text'][:max_chars]}"
        )
    return '\n\n---\n\n'.join(parts)


def parse_json_response(raw: str) -> tuple[dict, bool]:
    """Parse JSON from model response. Returns (parsed_dict, json_valid)."""
    # Strip markdown fences if model adds them despite instructions
    cleaned = re.sub(r'^```(?:json)?\s*', '', raw.strip(), flags=re.MULTILINE)
    cleaned = re.sub(r'```\s*$', '', cleaned, flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned), True
    except json.JSONDecodeError:
        return {
            'answer': raw[:300],
            'citations': [],
            'confidence': 0.0,
            'contradictions': [],
        }, False


def retrieve_context(query: str, top_k: int = 5) -> list[dict]:
    """Run full retrieval pipeline: hybrid retrieve → rerank → top-k."""
    candidates = hybrid.retrieve(query, top_k=20)
    reranked   = reranker.rerank(query, candidates)
    return reranked[:top_k]


print('Synthesis helpers defined.')

Synthesis helpers defined.


## 6. Model call functions

Pricing (as of May 2025, per million tokens):
- **Haiku 3.5:** $0.80 input / $4.00 output
- **GPT-4o mini:** $0.15 input / $0.60 output

In [9]:
# ── Haiku 3.5 ──────────────────────────────────────────────────────────
HAIKU_MODEL       = 'claude-haiku-4-5'
HAIKU_COST_INPUT  = 0.80   # $ per million input tokens
HAIKU_COST_OUTPUT = 4.00   # $ per million output tokens


def synthesise_haiku(query: str, chunks: list[dict]) -> dict:
    context = format_context(chunks)
    prompt  = SYNTHESIS_PROMPT.format(query=query, context=context)

    t0 = time.time()
    response = anthropic_client.messages.create(
        model=HAIKU_MODEL,
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}],
    )
    latency_ms = (time.time() - t0) * 1000

    raw            = response.content[0].text
    input_tokens   = response.usage.input_tokens
    output_tokens  = response.usage.output_tokens
    cost_usd       = (input_tokens / 1e6 * HAIKU_COST_INPUT) + (output_tokens / 1e6 * HAIKU_COST_OUTPUT)
    parsed, valid  = parse_json_response(raw)

    return {
        'model':         HAIKU_MODEL,
        'parsed':        parsed,
        'json_valid':    valid,
        'latency_ms':    round(latency_ms),
        'input_tokens':  input_tokens,
        'output_tokens': output_tokens,
        'cost_usd':      round(cost_usd, 6),
        'raw':           raw,
    }


# ── GPT-4o mini ────────────────────────────────────────────────────────
GPT_MODEL       = 'gpt-4o-mini'
GPT_COST_INPUT  = 0.15   # $ per million input tokens
GPT_COST_OUTPUT = 0.60   # $ per million output tokens


def synthesise_gpt(query: str, chunks: list[dict]) -> dict:
    context = format_context(chunks)
    prompt  = SYNTHESIS_PROMPT.format(query=query, context=context)

    t0 = time.time()
    response = openai_client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        max_tokens=512,
        temperature=0.0,
    )
    latency_ms = (time.time() - t0) * 1000

    raw            = response.choices[0].message.content
    input_tokens   = response.usage.prompt_tokens
    output_tokens  = response.usage.completion_tokens
    cost_usd       = (input_tokens / 1e6 * GPT_COST_INPUT) + (output_tokens / 1e6 * GPT_COST_OUTPUT)
    parsed, valid  = parse_json_response(raw)

    return {
        'model':         GPT_MODEL,
        'parsed':        parsed,
        'json_valid':    valid,
        'latency_ms':    round(latency_ms),
        'input_tokens':  input_tokens,
        'output_tokens': output_tokens,
        'cost_usd':      round(cost_usd, 6),
        'raw':           raw,
    }


print('Model call functions defined.')

Model call functions defined.


## 7. Run comparison

Each query: retrieve context once → call both models → record results.  
~20–40 seconds total.

In [10]:
print('Running model comparison...\n')

results: list[dict] = []

for q in EVAL_QUERIES:
    print(f"  {q['id']} [{q['type']:9s}] {q['query'][:60]}...")

    # Retrieve context once — shared for both models
    chunks = retrieve_context(q['query'])

    haiku_res = synthesise_haiku(q['query'], chunks)
    gpt_res   = synthesise_gpt(q['query'], chunks)

    results.append({
        'query_id':        q['id'],
        'query':           q['query'],
        'type':            q['type'],
        'expected_source': q['expected_source'],
        'haiku':           haiku_res,
        'gpt':             gpt_res,
        'chunks':          chunks,
    })

    h_conf = haiku_res['parsed'].get('confidence', '?')
    g_conf = gpt_res['parsed'].get('confidence', '?')
    print(f"    Haiku: {haiku_res['latency_ms']:>5}ms | JSON={str(haiku_res['json_valid']):5s} | conf={h_conf}")
    print(f"    GPT:   {gpt_res['latency_ms']:>5}ms | JSON={str(gpt_res['json_valid']):5s} | conf={g_conf}")

print(f'\nDone. {len(results)}/10 queries evaluated.')

Running model comparison...

  Q01 [factual  ] What load control licensing requirements does Ofgem propose ...
    Haiku:  2893ms | JSON=True  | conf=0.72
    GPT:   17687ms | JSON=True  | conf=0.9
  Q02 [factual  ] What enforcement powers will Ofgem have against unlicensed l...
    Haiku:  1628ms | JSON=True  | conf=0.25
    GPT:    5155ms | JSON=True  | conf=0.9
  Q03 [factual  ] What aggregate loan losses did UK banks face under the CBES ...
    Haiku:  1610ms | JSON=True  | conf=0.15
    GPT:    3100ms | JSON=True  | conf=0.2
  Q04 [factual  ] How did the CBES late policy transition scenario affect UK b...
    Haiku:  1244ms | JSON=True  | conf=0.25
    GPT:    4240ms | JSON=True  | conf=0.8
  Q05 [factual  ] What does the IEA project for peak global fossil fuel demand...
    Haiku:  2398ms | JSON=True  | conf=0.15
    GPT:    3629ms | JSON=True  | conf=0.9
  Q06 [factual  ] What annual clean energy investment does the IEA project wil...
    Haiku:  1416ms | JSON=True  | conf=0.2
 

## 8. Side-by-side results

**Score each answer 1–5 after reading it:**
- 5 = accurate, well-grounded citations, appropriate confidence
- 4 = mostly correct, minor imprecision
- 3 = partially correct or vague
- 2 = significant inaccuracies or unhelpful
- 1 = wrong, hallucinated, or refused when answer was available

For **negatives (Q09, Q10):** a good response has confidence < 0.4 and a clear "not in corpus" answer. A bad response has high confidence and an invented answer.

In [11]:
for r in results:
    print(f"\n{'='*80}")
    print(f"  {r['query_id']} [{r['type']}]: {r['query']}")
    if r['expected_source']:
        print(f"  Expected source: {r['expected_source']}")
    
    # Show top retrieved chunk for context
    if r['chunks']:
        top = r['chunks'][0]
        print(f"  Top chunk: {top['doc_id']} (score={top.get('rerank_score', 0):.3f})")
    
    print(f"{'='*80}")

    for model_key, label in [('haiku', 'HAIKU 3.5'), ('gpt', 'GPT-4o mini')]:
        res    = r[model_key]
        parsed = res['parsed']
        conf   = parsed.get('confidence', '?')
        cites  = parsed.get('citations', [])

        print(f"\n  [{label}]")
        print(f"  Latency: {res['latency_ms']}ms | Tokens: {res['input_tokens']}in/{res['output_tokens']}out | JSON: {res['json_valid']} | Confidence: {conf}")
        print(f"  Answer:")
        print(f"    {parsed.get('answer', '')[:250]}")
        if cites:
            print(f"  Citations ({len(cites)}):")
            for c in cites[:3]:
                print(f"    [{c.get('doc_id', '?')}] {c.get('passage', '')[:80]}")
        else:
            print(f"  Citations: (none)")


  Q01 [factual]: What load control licensing requirements does Ofgem propose for flexibility service providers?
  Expected source: OFGEM_SSES_2025
  Top chunk: OFGEM_SSES_2025 (score=4.411)

  [HAIKU 3.5]
  Latency: 2893ms | Tokens: 1181in/374out | JSON: True | Confidence: 0.72
  Answer:
    Ofgem proposes load control licensing requirements that include standard questions about company details and structure, suitability assessments confirming directors and key senior individuals are fit and proper to hold a licence, and operational capa
  Citations (3):
    [OFGEM_SSES_2025] A new load control licensing regime is being developed which will put in place r
    [OFGEM_SSES_2025] Standard questions (1.1-8.4): Details about the company, including structure and
    [OFGEM_SSES_2025] To ensure our approach is practical and delivers the programme's strategic ambit

  [GPT-4o mini]
  Latency: 17687ms | Tokens: 994in/268out | JSON: True | Confidence: 0.9
  Answer:
    Ofgem proposes a new load

## 9. Manual quality scores

**After reviewing the outputs above, fill in scores 1–5 for each model/query pair.**

Replace the `0`s in `MANUAL_SCORES` below, then run the cell.

In [12]:
#          Q01  Q02  Q03  Q04  Q05  Q06  Q07  Q08  Q09  Q10
MANUAL_SCORES = {
    'haiku': [ 4,   2,   3,   2,   1,   1,   2,   2,   5,   5 ],
    'gpt':   [ 4,   4,   3,   3,   5,   5,   3,   3,   5,   5 ],
}

# Validate
assert all(0 <= s <= 5 for s in MANUAL_SCORES['haiku']), 'Scores must be 0-5'
assert all(0 <= s <= 5 for s in MANUAL_SCORES['gpt']),   'Scores must be 0-5'

for model_key, label in [('haiku', 'Haiku 3.5'), ('gpt', 'GPT-4o mini')]:
    scores     = MANUAL_SCORES[model_key]
    factual    = scores[:6]
    synthesis  = scores[6:8]
    negatives  = scores[8:]
    filled     = sum(1 for s in scores if s > 0)
    avg_all    = sum(scores) / len(scores) if filled == 10 else 'TBD'
    avg_fact   = sum(factual) / 6 if all(factual) else 'TBD'
    print(f'{label}: avg={avg_all} | factual={avg_fact} | scores={scores} ({filled}/10 filled)')

if all(s > 0 for s in MANUAL_SCORES['haiku'] + MANUAL_SCORES['gpt']):
    print('\nAll scores filled — ready to run Section 10 (lock decision).')
else:
    print('\nFill in the 0s above after reviewing outputs, then re-run.')

Haiku 3.5: avg=2.7 | factual=2.1666666666666665 | scores=[4, 2, 3, 2, 1, 1, 2, 2, 5, 5] (10/10 filled)
GPT-4o mini: avg=4.0 | factual=4.0 | scores=[4, 4, 3, 3, 5, 5, 3, 3, 5, 5] (10/10 filled)

All scores filled — ready to run Section 10 (lock decision).


## 10. Cost & latency summary

In [13]:
haiku_lats    = [r['haiku']['latency_ms'] for r in results]
gpt_lats      = [r['gpt']['latency_ms']   for r in results]
haiku_costs   = [r['haiku']['cost_usd']   for r in results]
gpt_costs     = [r['gpt']['cost_usd']     for r in results]
haiku_json_ok = sum(1 for r in results if r['haiku']['json_valid'])
gpt_json_ok   = sum(1 for r in results if r['gpt']['json_valid'])

haiku_scores = MANUAL_SCORES['haiku']
gpt_scores   = MANUAL_SCORES['gpt']
haiku_avg_q  = sum(haiku_scores) / len(haiku_scores) if all(haiku_scores) else None
gpt_avg_q    = sum(gpt_scores)   / len(gpt_scores)   if all(gpt_scores)   else None

# Negative query confidence (lower is better)
neg_haiku_conf = [r['haiku']['parsed'].get('confidence', 1.0) for r in results if r['type'] == 'negative']
neg_gpt_conf   = [r['gpt']['parsed'].get('confidence', 1.0)   for r in results if r['type'] == 'negative']

rows = [
    ('Avg quality score (1-5)', f"{haiku_avg_q:.2f}" if haiku_avg_q else 'TBD', f"{gpt_avg_q:.2f}" if gpt_avg_q else 'TBD'),
    ('JSON compliance',         f'{haiku_json_ok}/10', f'{gpt_json_ok}/10'),
    ('Avg latency (ms)',        f'{statistics.mean(haiku_lats):.0f}', f'{statistics.mean(gpt_lats):.0f}'),
    ('p95 latency (ms)',        f'{sorted(haiku_lats)[int(0.95*len(haiku_lats))]:.0f}', f'{sorted(gpt_lats)[int(0.95*len(gpt_lats))]:.0f}'),
    ('Total cost (10 queries)', f'${sum(haiku_costs):.4f}', f'${sum(gpt_costs):.4f}'),
    ('Est. cost per 1k queries',f'${sum(haiku_costs)*100:.2f}', f'${sum(gpt_costs)*100:.2f}'),
    ('Neg. query avg conf',     f'{statistics.mean(neg_haiku_conf):.2f}' if neg_haiku_conf else 'N/A',
                                f'{statistics.mean(neg_gpt_conf):.2f}'   if neg_gpt_conf   else 'N/A'),
]

print(f"\n{'Metric':<30} {'Haiku 3.5':>12} {'GPT-4o mini':>12}")
print('-' * 56)
for metric, h_val, g_val in rows:
    print(f'{metric:<30} {h_val:>12} {g_val:>12}')

print('\nNote: neg. query avg confidence should be < 0.40 — high conf on negatives = hallucination risk.')


Metric                            Haiku 3.5  GPT-4o mini
--------------------------------------------------------
Avg quality score (1-5)                2.70         4.00
JSON compliance                       10/10        10/10
Avg latency (ms)                       1552         4700
p95 latency (ms)                       2893        17687
Total cost (10 queries)             $0.0147      $0.0025
Est. cost per 1k queries              $1.47        $0.25
Neg. query avg conf                    0.03         0.20

Note: neg. query avg confidence should be < 0.40 — high conf on negatives = hallucination risk.


## 11. Lock decision & write outputs

**Instructions:**
1. Set `WINNER_MODEL` to the winner below.
2. Fill in `DECISION_RATIONALE` (2–3 sentences).
3. Run the cell — it writes `model_selection.md` and updates `configs/config.yaml`.

In [14]:
# ── SET THESE BEFORE RUNNING ──────────────────────────────────────────
WINNER_MODEL = 'gpt-4o-mini'   # or 'gpt-4o-mini'
WINNER_ALIAS = 'GPT-4o mini'   # or 'GPT-4o mini'

DECISION_RATIONALE = (
    'GPT-4o mini substantially outperformed Haiku 4.5 on answer quality (4.0 vs 2.70 '
    'avg), correctly extracting specific figures from retrieved chunks — including peak '
    'fossil fuel demand timing and the $4.8 trillion NZE investment figure — that Haiku '
    'consistently refused to synthesise despite reranker scores above 4.5. '
    'GPT-4o mini is also 6x cheaper per query ($0.25 vs $1.47 per 1,000). '
    'Haiku is 3x faster on average (1552ms vs 4700ms), which is a genuine advantage, '
    'but CPIE serves research workflows where sub-5s synthesis latency is acceptable '
    'and answer quality is the primary success criterion. '
    'Winner: GPT-4o mini for CPIE v1.'
)
# ─────────────────────────────────────────────────────────────────────

haiku_avg_q   = sum(MANUAL_SCORES['haiku']) / 10 if all(MANUAL_SCORES['haiku']) else 0.0
gpt_avg_q     = sum(MANUAL_SCORES['gpt'])   / 10 if all(MANUAL_SCORES['gpt'])   else 0.0
haiku_lats    = [r['haiku']['latency_ms'] for r in results]
gpt_lats      = [r['gpt']['latency_ms']   for r in results]
haiku_costs   = [r['haiku']['cost_usd']   for r in results]
gpt_costs     = [r['gpt']['cost_usd']     for r in results]
haiku_json_ok = sum(1 for r in results if r['haiku']['json_valid'])
gpt_json_ok   = sum(1 for r in results if r['gpt']['json_valid'])

md_content = f"""# Model Selection — CPIE Synthesis Layer

**Decision date:** {time.strftime('%Y-%m-%d')}
**Winner:** {WINNER_ALIAS} (`{WINNER_MODEL}`)

---

## Evaluation Setup

- **Corpus:** 3 pilot PDFs — Ofgem SSES 2025, CBES Results 2022, IEA WEO 2025
- **Retrieval:** Hybrid RRF k=60 (BM25 + BAAI/bge-base-en-v1.5) → cross-encoder reranker → top-5 chunks
- **Queries:** 10 total — 6 factual (2 per doc), 2 cross-document synthesis, 2 out-of-corpus negatives
- **Scoring:** Manual 1–5 quality rating (answer accuracy + citation faithfulness + schema compliance)

---

## Results

| Metric | Haiku 3.5 | GPT-4o mini |
|---|---|---|
| Avg quality score (1–5) | {haiku_avg_q:.1f} | {gpt_avg_q:.1f} |
| JSON schema compliance | {haiku_json_ok}/10 | {gpt_json_ok}/10 |
| Avg latency | {statistics.mean(haiku_lats):.0f}ms | {statistics.mean(gpt_lats):.0f}ms |
| Est. cost per 1,000 queries | ${sum(haiku_costs)*100:.2f} | ${sum(gpt_costs)*100:.2f} |

---

## Decision Rationale

{DECISION_RATIONALE}

---

## Locked Settings

```yaml
synthesis:
  model: {WINNER_MODEL}
  max_tokens: 512
  temperature: 0.0
```

---

*Raw results: `data/eval/results/model_selection_raw.json`*
*Notebook: `notebooks/model_selection.ipynb`*
"""

# Write model_selection.md
md_path = ROOT / 'model_selection.md'
md_path.write_text(md_content, encoding='utf-8')
print(f'Written: {md_path}')

# Update configs/config.yaml — synthesis.model
config_path = ROOT / 'configs/config.yaml'
config_text = config_path.read_text(encoding='utf-8')
config_text = re.sub(
    r'(synthesis:\s*\n\s*model:\s*).*',
    f'\\g<1>{WINNER_MODEL}  # LOCKED — {WINNER_ALIAS} won model selection (Week 2 Step 6)',
    config_text,
)
config_text = re.sub(
    r'(\s*max_tokens:\s*)\d+',
    '\\g<1>512',
    config_text,
)
config_path.write_text(config_text, encoding='utf-8')
print(f'Updated: {config_path}')

# Save raw results
results_path = ROOT / 'data/eval/results/model_selection_raw.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
serializable = []
for i, r in enumerate(results):
    serializable.append({
        'query_id':        r['query_id'],
        'query':           r['query'],
        'type':            r['type'],
        'expected_source': r['expected_source'],
        'haiku': {k: v for k, v in r['haiku'].items() if k != 'raw'},
        'gpt':   {k: v for k, v in r['gpt'].items()   if k != 'raw'},
        'manual_scores': {
            'haiku': MANUAL_SCORES['haiku'][i],
            'gpt':   MANUAL_SCORES['gpt'][i],
        },
    })

with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(serializable, f, indent=2)
print(f'Saved:   {results_path}')

print(f'\nModel selection complete. Winner locked: {WINNER_ALIAS} ({WINNER_MODEL})')

Written: C:\AI_Engineering\cpie\model_selection.md
Updated: C:\AI_Engineering\cpie\configs\config.yaml
Saved:   C:\AI_Engineering\cpie\data\eval\results\model_selection_raw.json

Model selection complete. Winner locked: GPT-4o mini (gpt-4o-mini)


## 12. Cleanup test Chroma collection

In [15]:
dense.delete_collection()
print('Test Chroma collection deleted.')

Test Chroma collection deleted.
